# ⚙️ Notebook 04: Feature Pipeline & Data Preparation

> **Mục tiêu**: Trực quan hóa từng bước trong quá trình xử lý dữ liệu (Tuần 2). Notebook này sẽ giúp bạn nhìn thấy rõ input/output của từng bước thay vì chỉ chạy file `.py` ẩn bên dưới.


## 1. Load Data & Tách Train/Test (Temporal Split)
Thay vì random split, chúng ta dùng biến `month` để tách tập Train (tháng 0-5) và Test (tháng 6-7).


In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Thêm root path để import module
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_raw_data, temporal_split

# Load data
print("Đang load Base.csv...")
df_raw = load_raw_data("Base.csv")
display(df_raw.head())


Đang load Base.csv...
Loading raw data from /home/host/fraud_detection/data/raw/Base.csv...


,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,...,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0,0.3,0.986506,-1,25,40,0.006735,102.453711,AA,1059,...,0,1500.0,0,INTERNET,16.224843,linux,1,1,0,0
1,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,AD,1658,...,0,1500.0,0,INTERNET,3.363854,other,1,1,0,0
2,0,0.8,0.996707,9,14,40,0.012316,-1.490386,AB,1095,...,0,200.0,0,INTERNET,22.730559,windows,0,1,0,0
3,0,0.6,0.475100,11,14,30,0.006991,-1.863101,AB,3483,...,0,200.0,0,INTERNET,15.215816,linux,1,1,0,0
4,0,0.9,0.842307,-1,29,40,5.742626,47.152498,AA,2339,...,0,200.0,0,INTERNET,3.743048,other,0,1,0,0


In [2]:
# Chia Train/Test
df_train, df_test = temporal_split(df_raw, train_months=[0,1,2,3,4,5], test_months=[6,7])

print(f"Shape Train: {df_train.shape}")
print(f"Shape Test: {df_test.shape}")


Splitting data based on temporal feature (month)...
Train set: 794,989 rows (Months [0, 1, 2, 3, 4, 5])
Test set: 205,011 rows (Months [6, 7])
Shape Train: (794989, 32)
Shape Test: (205011, 32)


## 2. Feature Engineering
Xử lý Missing Values (-1), Scale biến số (StandardScaler) và Encode biến phân loại (OrdinalEncoder).


In [3]:
from src.feature_engineering import apply_feature_engineering

print("Applying Feature Engineering trên tập TRAIN...")
df_train_processed, preprocessor = apply_feature_engineering(df_train, is_train=True)

print("Kích thước sau xử lý:", df_train_processed.shape)
display(df_train_processed.head())


Applying Feature Engineering trên tập TRAIN...
Kích thước sau xử lý: (794989, 35)


,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,zip_count_4w,velocity_6h,velocity_24h,...,phone_home_valid,phone_mobile_valid,has_other_cards,foreign_request,keep_alive_session,prev_address_months_count_is_missing,current_address_months_count_is_missing,bank_months_count_is_missing,fraud_bool,month
0,-0.847913,1.676177,-0.217724,-0.705423,0.523928,-0.196880,4.578476,-0.600874,2.305362,2.012562,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0,0
1,0.866055,0.407216,-0.217724,0.007684,-1.130108,-0.196255,-0.477577,-0.023033,1.020297,0.471870,...,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0,0
2,0.866055,1.711249,-0.850011,-0.827988,0.523928,-0.195842,-0.508942,-0.566145,-0.556459,0.271930,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
3,0.180468,-0.082125,-0.795030,-0.827988,-0.303090,-0.196833,-0.527184,1.737498,2.748663,1.210931,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
4,1.208849,1.180395,-0.217724,-0.660853,0.523928,0.869760,1.871825,0.633910,0.482157,0.017349,...,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0,0


In [4]:
print("Applying Feature Engineering trên tập TEST (dùng chung preprocessor của Train)...")
df_test_processed, _ = apply_feature_engineering(df_test, is_train=False, preprocessor=preprocessor)

print("Kích thước sau xử lý:", df_test_processed.shape)


Applying Feature Engineering trên tập TEST (dùng chung preprocessor của Train)...
Kích thước sau xử lý: (205011, 35)


## 3. Class Imbalance (scale_pos_weight)
Dữ liệu cực kỳ mất cân bằng (fraud ~1.1%). Chúng ta tính trọng số `scale_pos_weight` để gán cho XGBoost/LightGBM ở Tuần 3.


In [5]:
from src.imbalance import calculate_scale_pos_weight

y_train = df_train_processed['fraud_bool'].values
spw = calculate_scale_pos_weight(y_train)

print(f"Tỷ lệ số ca âm / số ca dương = {spw:.2f}")
print("-> Ghi nhớ thông số này để config model!")


Tỷ lệ số ca âm / số ca dương = 96.53
-> Ghi nhớ thông số này để config model!


## 4. Lưu ra định dạng Parquet
Parquet lưu trữ metadata (tên cột, data types) và dung lượng nén nhỏ hơn CSV rất nhiều.


In [6]:
from src.data_loader import save_processed_data

# Dữ liệu đã được lưu bởi run_pipeline.py, code này mô phỏng lại
print("Lưu df_train_processed -> data/processed/train.parquet")
print("Lưu df_test_processed -> data/processed/test.parquet")


Lưu df_train_processed -> data/processed/train.parquet
Lưu df_test_processed -> data/processed/test.parquet
